# Advanced RAG: Document Splitting and FAISS

This notebook covers advanced RAG techniques including document chunking, overlap strategies, and using FAISS for efficient vector search.

## Learning Objectives
- Split long documents into manageable chunks
- Understand chunk overlap and its importance
- Use FAISS for high-performance vector search
- Compare different vector store implementations

## Related Theoretical Content
- [RAG Systems](../../notes/03-ai/README.md)
- [Vector Databases](../../notes/03-ai/05-vector_databases/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import Dependencies

Load all required libraries and configure API access.

In [ ]:
import os
from uuid import uuid4
from api_keys import get_api_key

# Configure environment
os.environ["OPENAI_API_KEY"] = get_api_key('openai')

print(" Environment configured")

## 1. Download Sample Documents

Create a knowledge base for testing document splitting and retrieval.

In [ ]:
import wikipedia

# Download cricket player biographies
players = [
    "Sachin Tendulkar", "Jacques Kallis", "AB De Villiers",
    "Matthew Hayden", "Brian Lara", "Virat Kohli",
    "Kumar Sangakkara", "Mahela Jayawardene"
]

data_dir = ".data/cricket_advanced"
os.makedirs(data_dir, exist_ok=True)

print("Downloading player biographies...")
for player in players:
    try:
        page = wikipedia.page(player, auto_suggest=False)
        content = page.summary

        filename = f"{player.replace(' ', '_')}.txt"
        filepath = os.path.join(data_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)

        print(f" {player}")
    except Exception as e:
        print(f"✗ {player}: {e}")

print(f"\n Documents saved to {data_dir}")

## 2. Load Documents with DirectoryLoader

**DirectoryLoader** automatically loads all files matching a pattern from a directory.

Benefits:
- Handles multiple file formats
- Preserves metadata (file paths, dates, etc.)
- Simplifies document ingestion

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all .txt files from directory
loader = DirectoryLoader(
    data_dir,
    glob="**/*.txt",
    loader_cls=TextLoader
)

docs = loader.load()

print(f" Loaded {len(docs)} documents")
print(f"\nSample document:")
print(f"Source: {docs[0].metadata['source']}")
print(f"Length: {len(docs[0].page_content)} characters")
print(f"Preview: {docs[0].page_content[:200]}...")

## 3. Document Splitting with Overlap

**Why split documents?**
- Long documents exceed LLM context limits
- Smaller chunks improve retrieval precision
- Only relevant sections are included in context

**Chunk Overlap:**
- Ensures context isn't lost at chunk boundaries
- 20% overlap (160 of 800 chars) is a common heuristic
- Prevents splitting sentences or paragraphs awkwardly

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter with 20% overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=160  # 20% of chunk_size
)

# Split documents into chunks
chunks = text_splitter.split_documents(docs)

print(f" Split {len(docs)} documents into {len(chunks)} chunks")
print(f"\nChunk examples:")
for i in range(min(3, len(chunks))):
    print(f"\nChunk {i+1}:")
    print(f"  Source: {chunks[i].metadata['source']}")
    print(f"  Length: {len(chunks[i].page_content)} chars")
    print(f"  Content: {chunks[i].page_content[:150]}...")

## 4. Visualize Chunk Overlap

Let's examine how overlapping works by looking at consecutive chunks.

In [ ]:
# Compare two consecutive chunks from the same document
if len(chunks) >= 2:
    chunk1 = chunks[8].page_content
    chunk2 = chunks[9].page_content

    print("Chunk 1 (last 100 chars):")
    print(chunk1[-100:])
    print("\n" + "="*60 + "\n")
    print("Chunk 2 (first 100 chars):")
    print(chunk2[:100])
    print("\nNotice the overlapping content between chunks!")

## 5. Initialize Embedding Model

Use the same efficient embedding model for consistency.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Get embedding dimension for FAISS setup
sample_embedding = embedding_model.embed_query("test")
embedding_length = len(sample_embedding)

print(f" Embedding model loaded (dimension: {embedding_length})")

## 6. Create FAISS Vector Store

**FAISS** (Facebook AI Similarity Search) is a highly optimized library for similarity search.

**Advantages over ChromaDB:**
- Extremely fast for large datasets (millions of vectors)
- Multiple index types optimized for different use cases
- Lower memory footprint
- Developed by Meta AI Research

**IndexFlatL2:**
- Exact search (no approximation)
- Uses L2 (Euclidean) distance
- Best for smaller datasets (<100k vectors)

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# Initialize FAISS vector store
vector_store = FAISS(
    embedding_function=embedding_model,
    index=faiss.IndexFlatL2(embedding_length),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# Generate unique IDs for each chunk
uuids = [str(uuid4()) for _ in range(len(chunks))]

# Add documents with IDs
print("Creating embeddings and building FAISS index...")
vector_store.add_documents(documents=chunks, ids=uuids)

print(f" FAISS vector store created with {len(chunks)} chunks")

## 7. Perform Advanced Similarity Search

Test the FAISS vector store with various queries.

In [ ]:
# Query the vector store
query = "What is the formula for positional encoding?"
results = vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
print(f"Top {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Source: {result.metadata['source']}")
    print(f"   Content: {result.page_content[:200]}...\n")

## 8. Complete RAG Pipeline with Chunked Documents

Implement the full RAG flow using chunked documents and FAISS.

In [ ]:
from langchain_groq import ChatGroq

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=get_api_key('groq')
)

def advanced_rag_query(question: str, k: int = 3) -> str:
    """
    Query the advanced RAG system.

    Args:
        question: User's question
        k: Number of chunks to retrieve

    Returns:
        LLM-generated answer
    """
    # Retrieve relevant chunks
    relevant_chunks = vector_store.similarity_search(question, k=k)

    # Combine chunk content
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    # Create prompt
    prompt = f'''
    You are an assistant that answers questions based on provided context.
    Generate an accurate and concise answer based solely on the context.
    If the context doesn't contain enough information, say "No relevant information found".

    User Query: {question}

    Context:
    {context}
    '''

    # Generate response
    response = llm.invoke(prompt)
    return response.content

# Test queries
test_questions = [
    "What are Sachin Tendulkar's major achievements?",
    "Which cricket player is known for their versatility?",
    "What records did Brian Lara set?"
]

for question in test_questions:
    answer = advanced_rag_query(question)
    print(f"Q: {question}")
    print(f"A: {answer}\n")
    print("="*60 + "\n")

## 9. Save and Load FAISS Index

FAISS indexes can be saved to disk for reuse, avoiding recomputation.

In [ ]:
# Save the vector store
save_path = ".data/faiss_index_cricket"
vector_store.save_local(save_path)
print(f" Vector store saved to {save_path}")

# Load the vector store
loaded_vector_store = FAISS.load_local(
    save_path,
    embedding_model,
    allow_dangerous_deserialization=True  # Required for loading pickled data
)
print(f" Vector store loaded from {save_path}")

# Test loaded store
test_results = loaded_vector_store.similarity_search("Virat Kohli", k=1)
print(f"\nTest query result: {test_results[0].page_content[:150]}...")

## Key Takeaways

1. **Document Chunking**: Essential for handling long documents
2. **Overlap Strategy**: Prevents context loss at boundaries (typically 10-20%)
3. **FAISS Performance**: Much faster than basic vector stores for large datasets
4. **Index Types**: Choose based on dataset size and accuracy requirements
5. **Persistence**: Save indexes to avoid recomputation

## Advanced RAG Techniques (Not Covered)

- **Hybrid Search**: Combine semantic search with keyword matching
- **Re-ranking**: Reorder results using cross-encoders
- **Query Expansion**: Generate multiple query variations
- **Metadata Filtering**: Filter by document attributes before search

## Next Steps

- [05-agentic-ai.ipynb](05-agentic-ai.ipynb): Build AI agents with tools and reasoning
- [06-agent-memory.ipynb](06-agent-memory.ipynb): Add persistent memory to agents